In [1]:
#imports
import pandas as pd
import numpy as np
import re
import plotly.express as px
import os

#Set option to see all columns
pd.set_option('display.max_columns', None)

In [2]:
infile = "/home/tarricol/AI_RCP/data/bh_data_clean_all.csv"
outdir = "./"
trim_type = "treshold_all"
include_water = True
export_reagent_df = True

# Add reagent type mapping (used in trimming and filtering)
reagent_cols = {
    "catalyst_name": "C",
    "reagent_1_name": "B",
    "solvent_1_name": "S",
    "additives_name_merged": "A",
    "solvent_2_name": "L", # Consistent mapping for solvent 2
    "contains_water": "W"
}

In [3]:
if not os.path.exists(infile):
        print(f"Error: Input file not found at {infile}")
        exit(1)

print(f"Loading data from: {infile}")
try:
        data = pd.read_csv(infile)
        if data.empty:
                print(f"Error: Input file {infile} is empty.")
                exit(1)
        # Attempt to determine reaction type robustly
        if "rxn_type" not in data.columns or data["rxn_type"].isnull().all():
                print("Warning: 'rxn_type' column missing or empty. Cannot determine reaction type automatically.")
                rxn_type = "UnknownReaction" # Assign a default name
        else:
                # Use the first non-null value as representative type
                rxn_type = data["rxn_type"].dropna().iloc[0] if not data["rxn_type"].dropna().empty else "UnknownReaction"
        print(f"Detected reaction type: {rxn_type}")
except Exception as e:
    print(f"Error loading or reading CSV file {infile}: {e}")
    exit(1)


Loading data from: /home/tarricol/AI_RCP/data/bh_data_clean_all.csv
Detected reaction type: Buchwald-Hartwig


In [4]:
data[data['rxn_id'].str.startswith("ELN032036-334")]

,rxn_id,rxn_type,rxn_date,temperature_deg_c,time_h,startingmat_1_name,startingmat_1_smiles,startingmat_1_eq,startingmat_2_name,startingmat_2_smiles,startingmat_2_eq,reagent_1_name,reagent_1_smiles,reagent_1_eq,catalyst_name,catalyst_smiles,catalyst_eq,solvent_1_name,solvent_1_smiles,solvent_1_fraction,startingmat_1_area%,product_1_name,product_1_smiles,product_1_area%,suff_yield,contains_water,additives_name_merged,additives_smiles_merged,additives_fraction_merged,YieldCategory
14910,ELN032036-334_1_1_A1,Buchwald-Hartwig,10/02/2023,80.0,3.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.336119,NaOtBu,[Na+].CC(C)(C)[O-],1.461854,BINAP,CS(O[Pd-2]1([P+](C2=CC=CC=C2)(C3=CC=CC=C3)C4=C...,0.078620,PhMe,Cc1ccccc1,1.0,0.905667,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.000000,0,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",VLowYield
14911,ELN032036-334_1_1_A2,Buchwald-Hartwig,10/02/2023,80.0,3.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.098215,NaOtBu,[Na+].CC(C)(C)[O-],1.174952,P(tBu)3,Cl[Pd](C/C=C/C)[P](C(C)(C)C)(C(C)(C)C)C(C)(C)C,0.058141,PhMe,Cc1ccccc1,1.0,0.005636,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.315346,1,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",MidYield
14912,ELN032036-334_1_1_A3,Buchwald-Hartwig,10/02/2023,80.0,3.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.450811,NaOtBu,[Na+].CC(C)(C)[O-],1.533258,meCgPPh,CS(=O)(=O)O[Pd]c1ccccc1-c2ccccc2N.CC34CC5(C)OC...,0.080866,PhMe,Cc1ccccc1,1.0,0.811379,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.000000,0,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",VLowYield
14913,ELN032036-334_1_1_A4,Buchwald-Hartwig,10/02/2023,80.0,3.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.382974,NaOtBu,[Na+].CC(C)(C)[O-],1.515696,Triisobutyl-Phosphatrane,CC(CN1CCN2CCN([P]1([Pd](C3=CC=C(C(F)(F)F)C=C3)...,0.076091,PhMe,Cc1ccccc1,1.0,0.002751,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.059157,1,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",LowYield
14914,ELN032036-334_1_1_A5,Buchwald-Hartwig,10/02/2023,80.0,3.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.348580,NaOtBu,[Na+].CC(C)(C)[O-],1.457893,BippyPhos,C=CC[Pd](OS(=O)(C(F)(F)F)=O)[P](C1=CC=NN1C2=C(...,0.080303,PhMe,Cc1ccccc1,1.0,0.279998,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.113413,1,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",LowYield
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16800,ELN032036-334_14_1_H8,Buchwald-Hartwig,09/08/2023,80.0,16.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.337065,KHMDS,C[Si](C)(C)[N-][Si](C)(C)C.[K+],1.930325,QPhos,CC=C[CH2-].CC(C)(C)P(C1=C[CH-]C=C1)C(C)(C)C.C1...,0.080720,PhMe,Cc1ccccc1,1.0,0.065883,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.175023,1,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",LowYield
16801,ELN032036-334_14_1_H9,Buchwald-Hartwig,09/08/2023,80.0,16.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.347565,KHMDS,C[Si](C)(C)[N-][Si](C)(C)C.[K+],2.034190,XantPhos,CC1(C)C2=CC=CC3=C2OC4=C1C=CC=C4[P]([Pd](Cl)(CC...,0.072125,PhMe,Cc1ccccc1,1.0,0.593014,Syn-dimethylpiperidinylbiphenyl,C[C@@H]1CCC[C@H](C)N1c1ccc(cc1)-c1ccccc1,0.007868,0,False,NoAdditive,NoAdditive,"frozenset({nan, nan})",VLowYield
16802,ELN032036-334_14_1_H10,Buchwald-Hartwig,09/08/2023,80.0,16.0,bromobiphenyl,Brc1ccc(cc1)-c1ccccc1,1,"syn-2,6-dimethylpiperidine",C[C@@H]1CCC[C@H](C)N1,1.369068,KHMDS,C[Si](C)(C)[N-][Si](C)(C)C.[K+],2.035572,CPhos,CN(C)c1cccc(N(C)C)c1c2ccccc2P(c3ccccc3)C(C)(C)...,0.092892,PhMe,Cc1ccccc

------------------------------
# Train performance comparison BH

In [5]:
#imports
import pandas as pd
import numpy as np
import re
import plotly.express as px
import os

from scipy.stats import wilcoxon
from itertools import combinations

#Set option to see all columns
pd.set_option('display.max_columns', None)

In [6]:
bh_all_baseline = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_baseline_bh_all_20250806-120950_detailed.csv')
bh_all_emb_cb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_all_20250806-120959_detailed.csv')
bh_all_emb_sasa = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_all_20250806-120950_detailed.csv')
bh_all_emb_sec = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_all_20250806-121008_detailed.csv')
bh_all_emb_all = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_all_20250806-121010_detailed.csv')
bh_all_emb_xtb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_all_20250802-001330_detailed.csv') #Something weird happened here
bh_all_rxnfp = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_rxnfp_bh_all_20250806-121005_detailed.csv')
bh_all_seq = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_bh_all_20250806-121007_detailed.csv')
bh_all_seq_emb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_emb_bh_all_20250806-120957_detailed.csv')

bh_pos_baseline = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_baseline_bh_positive_20250806-120949_detailed.csv')
bh_pos_emb_all = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_positive_20250806-120953_detailed.csv')
bh_pos_rxnfp = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_rxnfp_bh_positive_20250806-121011_detailed.csv')
bh_pos_seq = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_bh_positive_20250806-121006_detailed.csv')
bh_pos_seq_emb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_emb_bh_positive_20250806-121010_detailed.csv')




temperatures_bh = {
    'bh_all_baseline': 0.992,
    'bh_all_emb_cb': 0.852,
    'bh_all_emb_sasa': 0.602,
    'bh_all_emb_sec': 1.007,
    'bh_all_emb_all': 1.185,
    'bh_all_emb_xtb': 1.256,
    'bh_all_rxnfp': 1.165,
    'bh_all_seq': 1.134,
    'bh_all_seq_emb': 1.496, 
    'bh_pos_rxnfp': 1.308,
    'bh_pos_baseline': 0.669,
    'bh_pos_emb_all': 1.572,
    'bh_pos_seq': 1.457,
    'bh_pos_seq_emb': 1.216,
}




In [7]:
#Load random-frequency results
bh_all_random_baselines = pd.read_csv('/home/tarricol/AI_RCP/best_results/baselines_baseline_bh_all_20250807-165715.csv')
bh_positive_random_baselines = pd.read_csv('/home/tarricol/AI_RCP/best_results/baselines_baseline_bh_positive_20250807-170011.csv')

bh_all_random_baseline_random = bh_all_random_baselines[(bh_all_random_baselines['baseline'] == 'random')]
bh_all_random_baseline_structured_random = bh_all_random_baselines[(bh_all_random_baselines['baseline'] == 'structured_random')]
bh_all_random_baseline_most_frequent = bh_all_random_baselines[(bh_all_random_baselines['baseline'] == 'most_frequent')]
bh_all_random_baseline_freq_chain = bh_all_random_baselines[(bh_all_random_baselines['baseline'] == 'freq_chain')]

bh_pos_random_baseline_random = bh_positive_random_baselines[(bh_positive_random_baselines['baseline'] == 'random')]
bh_pos_random_baseline_structured_random = bh_positive_random_baselines[(bh_positive_random_baselines['baseline'] == 'structured_random')]
bh_pos_random_baseline_most_frequent = bh_positive_random_baselines[(bh_positive_random_baselines['baseline'] == 'most_frequent')]
bh_pos_random_baseline_freq_chain = bh_positive_random_baselines[(bh_positive_random_baselines['baseline'] == 'freq_chain')]

In [8]:
bh_pos_random_baseline_freq_chain

,run_idx,T,baseline,split,accuracy,macro_recall,micro_recall
3,0,1,freq_chain,pos,0.035714,0.000616,0.000614
7,0,10,freq_chain,pos,0.285714,0.011662,0.009515
11,0,50,freq_chain,pos,0.678571,0.062165,0.039902
15,0,100,freq_chain,pos,0.714286,0.096092,0.084101
19,0,500,freq_chain,pos,0.892857,0.342493,0.250767
23,0,1000,freq_chain,pos,0.928571,0.477871,0.340393
27,1,1,freq_chain,pos,0.071429,0.001064,0.001535
31,1,10,freq_chain,pos,0.214286,0.011454,0.010129
35,1,50,freq_chain,pos,0.678571,0.048645,0.034991
39,1,100,freq_chain,pos,0.714286,0.090637,0.077655


In [9]:
bh_all_baseline

,run_index,overall_performance,accuracy_positive_T1,macro_recall_positive_T1,micro_recall_positive_T1,diversity_positive_T1,accuracy_negative_T1,macro_recall_negative_T1,micro_recall_negative_T1,diversity_negative_T1,inter_diversity_negative_T1,accuracy_positive_T10,macro_recall_positive_T10,micro_recall_positive_T10,diversity_positive_T10,accuracy_negative_T10,macro_recall_negative_T10,micro_recall_negative_T10,diversity_negative_T10,inter_diversity_negative_T10,accuracy_positive_T50,macro_recall_positive_T50,micro_recall_positive_T50,diversity_positive_T50,accuracy_negative_T50,macro_recall_negative_T50,micro_recall_negative_T50,diversity_negative_T50,inter_diversity_negative_T50,accuracy_positive_T100,macro_recall_positive_T100,micro_recall_positive_T100,diversity_positive_T100,accuracy_negative_T100,macro_recall_negative_T100,micro_recall_negative_T100,diversity_negative_T100,inter_diversity_negative_T100,accuracy_positive_T500,macro_recall_positive_T500,micro_recall_positive_T500,diversity_positive_T500,accuracy_negative_T500,macro_recall_negative_T500,micro_recall_negative_T500,diversity_negative_T500,inter_diversity_negative_T500,accuracy_positive_T1000,macro_recall_positive_T1000,micro_recall_positive_T1000,diversity_positive_T1000,accuracy_negative_T1000,macro_recall_negative_T1000,micro_recall_negative_T1000,diversity_negative_T1000,inter_diversity_negative_T1000
0,0,0.568890,0.000000,0.000000,0.000000,1.0,0.096774,0.002353,0.001341,1.0,1.0,0.250000,0.006570,0.016650,9.892857,0.451613,0.019022,0.013414,9.903226,9.903226,0.714286,0.081250,0.066275,47.178571,0.806452,0.059969,0.041806,47.580645,47.580645,0.821429,0.141975,0.104146,90.285714,0.838710,0.106335,0.075341,90.967742,90.967742,0.892857,0.321806,0.239634,345.000000,0.967742,0.380646,0.237872,344.838710,344.838710,0.892857,0.429873,0.322233,557.678571,0.967742,0.484961,0.315672,556.483871,556.483871
1,1,0.582598,0.035714,0.000924,0.000979,1.0,0.000000,0.000000,0.000000,1.0,1.0,0.214286,0.006926,0.004897,9.857143,0.419355,0.013445,0.006483,9.967742,9.967742,0.750000,0.076867,0.036565,48.607143,0.838710,0.077700,0.031970,48.774194,48.774194,0.821429,0.120497,0.067581,93.571429,0.838710,0.105392,0.061704,94.677419,94.677419,0.892857,0.337686,0.236043,389.964286,0.935484,0.321517,0.231165,395.322581,395.322581,0.928571,0.469387,0.337251,664.321429,0.967742,0.464892,0.327744,680.419355,680.419355
2,2,0.571661,0.035714,0.001232,0.000653,1.0,0.096774,0.001022,0.001341,1.0,1.0,0.214286,0.009488,0.010447,9.928571,0.258065,0.007608,0.005589,9.935484,9.935484,0.678571,0.040601,0.037545,48.142857,0.709677,0.044067,0.029958,48.645161,48.645161,0.821429,0.087173,0.057460,92.892857,0.903226,0.109479,0.055444,93.935484,93.935484,0.928571,0.363326,0.202416,378.857143,0.935484,0.303856,0.202996,386.645161,386.645161,0.928571,0.452234,0.293830,641.392857,0.967742,0.476610,0.310977,652.870968,652.870968
3,3,0.598736,0.000000,0.000000,0.000000,1.0,0.064516,0.001681,0.001118,1.0,1.0,0.392857,0.013299,0.010774,9.857143,0.419355,0.016676,0.011178,9.741935,9.741935,0.750000,0.081426,0.058766,47.285714,0.774194,0.066703,0.054773,47.354839,47.354839,0.857143,0.125165,0.103167,89.857143,0.838710,0.139765,0.088531,91.225806,91.225806,0.964286,0.388455,0.270323,339.071429,0.967742,0.400618,0.250838,346.677419,346.677419,0.964286,0.460842,0.346066,554.714286,1.000000,0.492139,0.329086,566.516129,566.516129
4,4,0.583030,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,1.0,1.0,0.178571,0.003083,0.004571,9.928571,0.258065,0.007228,0.005589,10.000000,10.000000,0.392857,0.020428,0.021221,48.750000,0.709677,0.102093,0.021239,48.677419,48.677419,0.714286,0.072333,0.056481,94.571429,0.903226,0.146326,0.053432,94.903226,94.903226,0.857143,0.257684,0.194580,401.500000,0.967742,0.316034,0.200984,406.419355,406.419355,0.964286,0.444109,0.313092,699.678571,1.000000,0.471080,0.305611,707.322581,707.322581
5,5,0.598372,0.035714,0.000564,0.000979,1.0,0.096774,0.001767,0.000894,1.0,1.0,0.285714,0.00

In [10]:
def get_model_performance_data(model, model_name, metric, t):
    """
    Helper function to extract performance data for a given model, metric, and T value.
    --- CORRECTED to handle RxnFP ---
    """
    # Handle baseline models that have 'split' and 'T' columns
    if model_name in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
        model_T = model[model['T'] == t]
        if metric.endswith('positive'):
            return model_T[model_T['split'] == 'pos'][metric[:-9]].values
        elif metric.endswith('negative'):
            return model_T[model_T['split'] == 'neg'][metric[:-9]].values
            
    # Handle other models (e.g., Seq, Emb, and the special case RxnFP)
    else:
        # --- FIX ---
        # For RxnFP, performance is constant. Always use T=1 data regardless of the requested 't'.
        # For all other models, use the given 't'.
        effective_t = 1 if model_name == 'RxnFP' else t
        metric_col_name = f'{metric}_T{effective_t}'
        
        if metric_col_name in model:
            return model[metric_col_name].values

    # Return an empty array if no data is found for the given conditions
    return np.array([])


def plot_wilcoxon_results(best_models, model_names, T, metrics, alpha=0.05):
    """
    Performs pairwise Wilcoxon tests and plots the results as a 2x3 grid of matrices.
    Green (i,j): model i is significantly better than model j.
    Red (j,i): model j is significantly worse than model i.
    White: No significant difference.
    P-values are displayed in the colored cells.
    """
    print("--- 🔬 Generating Wilcoxon Test Result Matrices ---")
    
    for metric, pretty_name in metrics:
        # Create a 2x3 subplot grid
        fig = make_subplots(
                rows=2, cols=3,
                subplot_titles=[f"T={t}" for t in T],
                y_title="",
                x_title="",
                horizontal_spacing=0.12,  # Increased space between columns
                vertical_spacing=0.15     # Increased space between rows
            )
        
        for t_idx, t in enumerate(T):
            n_models = len(model_names)
            results_matrix = np.zeros((n_models, n_models))
            p_value_text = np.full((n_models, n_models), "", dtype=object)

            for i, j in combinations(range(n_models), 2):
                perf_i = get_model_performance_data(best_models[i], model_names[i], metric, t)
                perf_j = get_model_performance_data(best_models[j], model_names[j], metric, t)

                if perf_i.size > 1 and perf_i.size == perf_j.size:
                    try:
                        # Use 'greater' and 'less' to get one-sided p-values if desired,
                        # but two-sided is standard for detecting any difference.
                        stat, p_value = wilcoxon(perf_i, perf_j, alternative='two-sided')
                        if p_value < alpha:
                            p_text = f"{p_value:.2g}" # Format p-value for display
                            mean_i = np.mean(perf_i)
                            mean_j = np.mean(perf_j)
                            if mean_i > mean_j:
                                results_matrix[i, j] = 1
                                results_matrix[j, i] = -1
                                p_value_text[i, j] = p_text
                                p_value_text[j, i] = p_text
                            elif mean_j > mean_i:
                                results_matrix[j, i] = 1
                                results_matrix[i, j] = -1
                                p_value_text[j, i] = p_text
                                p_value_text[i, j] = p_text
                    except ValueError:
                        pass # Wilcoxon test fails if all differences are zero

            # Calculate subplot row and column
            row = t_idx // 3 + 1
            col = t_idx % 3 + 1

            fig.add_trace(
                go.Heatmap(
                    z=results_matrix,
                    x=model_names,
                    y=model_names,
                    text=p_value_text, # Add the text matrix
                    texttemplate="%{text}", # Display the text
                    textfont={"size":10, "color":"white"}, # Style the text
                    colorscale=[[0, 'red'], [0.5, 'white'], [1.0, 'green']],
                    zmin=-1, zmax=1, showscale=False
                ),
                row=row, col=col
            )
        
        fig.update_layout(
            title_text=f"<b>Pairwise Significance Matrix for: {pretty_name}</b>",
            height=800, width=1200,
            xaxis_tickangle=-60 # Angle ticks on all subplots at once
        )
        fig.show()


import numpy as np
from itertools import combinations
from scipy.stats import wilcoxon
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Mapping needed to pull values for Frequency Chain from data_freq_baselines
_DIVERSITY_FREQCHAIN_MAPPING = {
    'diversity_positive': ('avg_diversity_pos', 'pos'),
    'diversity_negative': ('avg_diversity_neg', 'neg'),
    'inter_diversity_negative': ('avg_inter_diversity', 'pos'),
}


def get_diversity_performance_data(model_df: pd.DataFrame,
                                     model_name: str,
                                     metric_key: str,
                                     t_value: int) -> np.ndarray:
    """
    Returns per-run values for the given diversity metric and T for a specific model.
    --- CORRECTED to handle RxnFP ---
    """
    if model_name == 'Frequency Chain':
        col_name, required_split = _DIVERSITY_FREQCHAIN_MAPPING[metric_key]
        subset = model_df[
            (model_df['baseline'] == 'freq_chain') &
            (model_df['split'] == required_split) &
            (model_df['T'] == t_value)
        ]
        return subset[col_name].dropna().to_numpy(dtype=float)

    # --- FIX ---
    # For RxnFP, diversity is constant. Use the T=1 column for any requested t_value
    # and return the full array of values to allow for a valid statistical test.
    if model_name.lower() == 'rxnfp':
        column_name = f"{metric_key}_T1"
        if column_name in model_df.columns:
            return model_df[column_name].dropna().to_numpy(dtype=float)
        # Fallback if T1 column is missing for some reason
        return np.array([], dtype=float)

    # For all other models, construct the column name with the given t_value
    column_name = f"{metric_key}_T{t_value}"
    if column_name in model_df.columns:
        return model_df[column_name].dropna().to_numpy(dtype=float)

    return np.array([], dtype=float)



def plot_wilcoxon_diversity_results(best_models: list,
                                    model_names: list,
                                    T: list,
                                    metrics: list[tuple[str, str]] | None = None,
                                    alpha: float = 0.05) -> None:
    """
    Performs pairwise Wilcoxon tests for diversity metrics and plots 2x3 heatmaps over T.

    Args:
        best_models: list of DataFrames corresponding to `model_names` order. For
            'Frequency Chain', pass the `data_freq_baselines` DataFrame.
        model_names: list of display names (e.g., ['RxnFP','Baseline','Emb','Seq','Seq+Emb','Frequency Chain']).
        T: list of T values (e.g., [1,10,50,100,500,1000]).
        metrics: list of tuples (metric_key, pretty_name). If None, uses default
            diversity metrics.
        alpha: significance level for Wilcoxon test.
    """
    if metrics is None:
        metrics = [
            ('diversity_positive', 'Diversity (Positive)'),
            ('diversity_negative', 'Diversity (Negative)'),
            ('inter_diversity_negative', 'Inter-Diversity'),
        ]

    print("--- 🔬 Generating Wilcoxon Test Result Matrices for Diversity Metrics ---")

    for metric_key, pretty_name in metrics:
        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=[f"T={t}" for t in T],
            y_title="",
            x_title="",
            horizontal_spacing=0.12,
            vertical_spacing=0.15
        )

        for t_idx, t_value in enumerate(T):
            num_models = len(model_names)
            results_matrix = np.zeros((num_models, num_models))
            p_value_text = np.full((num_models, num_models), "", dtype=object)

            for i, j in combinations(range(num_models), 2):
                perf_i = get_diversity_performance_data(best_models[i], model_names[i], metric_key, t_value)
                perf_j = get_diversity_performance_data(best_models[j], model_names[j], metric_key, t_value)

                if perf_i.size > 1 and perf_i.size == perf_j.size:
                    try:
                        stat, p_value = wilcoxon(perf_i, perf_j, alternative='two-sided')
                        if p_value < alpha:
                            p_text = f"{p_value:.2g}"
                            mean_i = float(np.mean(perf_i))
                            mean_j = float(np.mean(perf_j))
                            if mean_i > mean_j:
                                results_matrix[i, j] = 1
                                results_matrix[j, i] = -1
                                p_value_text[i, j] = p_text
                                p_value_text[j, i] = p_text
                            elif mean_j > mean_i:
                                results_matrix[j, i] = 1
                                results_matrix[i, j] = -1
                                p_value_text[j, i] = p_text
                                p_value_text[i, j] = p_text
                    except ValueError:
                        # Happens if all differences are zero; skip
                        pass

            row = t_idx // 3 + 1
            col = t_idx % 3 + 1
            fig.add_trace(
                go.Heatmap(
                    z=results_matrix,
                    x=model_names,
                    y=model_names,
                    text=p_value_text,
                    texttemplate="%{text}",
                    textfont={"size": 10, "color": "white"},
                    colorscale=[[0, 'red'], [0.5, 'white'], [1.0, 'green']],
                    zmin=-1, zmax=1, showscale=False
                ),
                row=row, col=col
            )

        fig.update_layout(
            title_text=f"<b>Pairwise Significance Matrix for Diversity: {pretty_name}</b>",
            height=800, width=1200,
            xaxis_tickangle=-60
        )
        fig.show()

In [11]:
T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go
import numpy as np

# Define the models and their best results (inverted order)
model_names = ['Random','RxnFP', 'Structured Random', 'Most Frequent','Frequency Chain', 'Baseline', 'Emb', 'Seq', 'Seq+Emb']
best_models = [bh_all_random_baseline_random, bh_all_rxnfp, bh_all_random_baseline_structured_random, bh_all_random_baseline_most_frequent, bh_all_random_baseline_freq_chain, bh_all_baseline, bh_all_emb_all, bh_all_seq, bh_all_seq_emb]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['bh_all_random_baseline_random', 'bh_all_rxnfp', 'bh_all_random_baseline_structured_random', 'bh_all_random_baseline_most_frequent', 'bh_all_random_baseline_freq_chain', 'bh_all_baseline', 'bh_all_emb_all', 'bh_all_seq', 'bh_all_seq_emb']

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d'
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('accuracy_negative', 'Accuracy (Neg)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)'),
    ('micro_recall_negative', 'Micro Recall (Neg)'),
    ('macro_recall_negative', 'Macro Recall (Neg)')
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
                model_T = model[model['T'] == t]
                if metric[-8:] == 'positive':
                    mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                elif metric[-8:] == 'negative':
                    mean_val = np.mean(model_T[model_T['split'] == 'neg'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'neg'][metric[:-9]])
                else:
                    print(f"Metric {metric} not found")
                    
            else:
                metric_col_name = f'{metric}_T{t}'
                if model_names[m_idx] == 'RxnFP':
                    # RxnFP only has T=1, so for other T set to 0
                    if t == 1:
                        mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                        std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
                    else:
                        mean_val = 0
                        std_val = 0
                else:
                    mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                    std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)

# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.05,
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=250 * n_metrics,
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for BH all dataset",
    legend_title_text="Model",
    showlegend=True
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


/scratch/site/u/tarricol/conda/envs/AI_RCP_env_plots/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning:

invalid value encountered in scalar divide



In [12]:

T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go
import numpy as np

# Define the models and their best results (inverted order)
model_names = ['Random','RxnFP', 'Structured Random', 'Most Frequent','Frequency Chain', 'Baseline', 'Emb', 'Seq', 'Seq+Emb']
best_models = [bh_pos_random_baseline_random, bh_pos_rxnfp, bh_pos_random_baseline_structured_random, bh_pos_random_baseline_most_frequent, bh_pos_random_baseline_freq_chain, bh_pos_baseline, bh_pos_emb_all, bh_pos_seq, bh_pos_seq_emb]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['bh_pos_random_baseline_random', 'bh_pos_rxnfp', 'bh_pos_random_baseline_structured_random', 'bh_pos_random_baseline_most_frequent', 'bh_pos_random_baseline_freq_chain', 'bh_pos_baseline', 'bh_pos_emb_all', 'bh_pos_seq', 'bh_pos_seq_emb']

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d'
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)'),
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
                try:
                    model_T = model[model['T'] == t]
                    if metric[-8:] == 'positive':
                        mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                        std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                except KeyError:
                    mean_val = 0
                    std_val = 0
            
            elif model_names[m_idx] == 'RxnFP':
                # RxnFP only has T=1, so for other T set to 0
                if t == 1:
                    mean_val = np.mean(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
                    std_val = np.std(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
                else:
                    mean_val = 0
                    std_val = 0
            else:
                mean_val = np.mean(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
                std_val = np.std(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)

# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.15, # Reduced vertical spacing
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=250 * n_metrics, #Reduced height
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for BH positive dataset",
    legend_title_text="Model",
    showlegend=True
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


In [13]:
# Line plot for diversity_pos and diversity_neg with uncertainties

import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Define new models because there was an error in the previous code
bh_all_rxnfp_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_rxnfp_bh_all_20250812-195137_detailed.csv')
bh_all_baseline_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_baseline_bh_all_20250812-195155_detailed.csv')
bh_all_emb_all_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_bh_all_20250812-195200_detailed.csv')
bh_all_seq_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_bh_all_20250812-195158_detailed.csv')
bh_all_seq_emb_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_emb_bh_all_20250812-195147_detailed.csv')

data_freq_baselines = pd.read_csv('/home/tarricol/AI_RCP/experiment_4_results/experiment4_emb_bh_all_20250812-170808.csv')

# Define the models and their best results (inverted order)
model_names = ['RxnFP', 'Baseline', 'Emb', 'Seq', 'Seq+Emb', 'Frequency Chain']
best_models = [bh_all_rxnfp_new, bh_all_baseline_new, bh_all_emb_all_new, bh_all_seq_new, bh_all_seq_emb_new, data_freq_baselines]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['bh_all_rxnfp_new', 'bh_all_baseline_new', 'bh_all_emb_all_new', 'bh_all_seq_new', 'bh_all_seq_emb_new', 'data_freq_baselines']

T = [1, 10, 50, 100, 500, 1000]

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d'
}


n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

temperatures_bh = {
    'bh_all_baseline_new': 0.992,
    'bh_all_emb_cb': 0.852,
    'bh_all_emb_sasa': 0.602,
    'bh_all_emb_sec': 1.007,
    'bh_all_emb_all_new': 1.185,
    'bh_all_emb_xtb': 1.256,
    'bh_all_rxnfp_new': 1.165,
    'bh_all_seq_new': 1.134,
    'bh_all_seq_emb_new': 1.496, 
    'bh_pos_rxnfp': 1.308,
    'bh_pos_baseline': 0.669,
    'bh_pos_emb_all': 1.572,
    'bh_pos_seq': 1.457,
    'bh_pos_seq_emb': 1.216,
    'data_freq_baselines': 1.000,
}


# Metric mapping
# - 'diversity_positive' comes from rows with split == 'pos' and column 'avg_diversity_pos'
# - 'diversity_negative' comes from rows with split == 'neg' and column 'avg_diversity_neg'
# - 'inter_diversity_negative' uses column 'avg_inter_diversity' available on 'pos' rows
metric_to_freqchain_source = {
    'diversity_positive': ('avg_diversity_pos', 'pos'),
    'diversity_negative': ('avg_diversity_neg', 'neg'),
    'inter_diversity_negative': ('avg_inter_diversity', 'pos'),
}

# Display names
diversity_metrics = ['diversity_positive', 'diversity_negative', 'inter_diversity_negative']
diversity_metric_names = ['Diversity (Positive)', 'Diversity (Negative)', 'Inter-Diversity']

# Helper to compute mean/std for Frequency Chain for a given metric and T
def compute_freq_chain_stats(df: pd.DataFrame, metric_key: str, t_value: int) -> tuple[float, float]:
    col_name, required_split = metric_to_freqchain_source[metric_key]
    subset = df[(df['baseline'] == 'freq_chain') & (df['split'] == required_split) & (df['T'] == t_value)]
    values = subset[col_name].dropna()
    if values.empty:
        return 0.0, 0.0
    if len(values) == 1:
        return float(values.iloc[0]), 0.0
    return float(values.mean()), float(values.std())

# Build figure
fig_div = make_subplots(rows=3, cols=1, vertical_spacing=0.12, subplot_titles=diversity_metric_names)

for idx, (metric, metric_name) in enumerate(zip(diversity_metrics, diversity_metric_names)):
    for m_idx, model_name in enumerate(model_names):
        model_df = best_models[m_idx]
        model_key = model_keys[m_idx]
        means: list[float] = []
        stds: list[float] = []

        temperature = round(temperatures_bh.get(model_key, None), 2) if model_key in temperatures_bh else None
        legend_name = f"{model_name} ({temperature})" if temperature is not None else model_name

        if model_name == 'Frequency Chain':
            for t in T:
                mean_val, std_val = compute_freq_chain_stats(model_df, metric, t)
                means.append(mean_val)
                stds.append(std_val)
        elif model_name.lower() == 'rxnfp':
            means = [1.0 for _ in T]
            stds = [0.0 for _ in T]
        else:
            for t in T:
                metric_col_name = f'{metric}_T{t}'
                if metric_col_name in model_df.columns:
                    col_values = model_df[metric_col_name].dropna()
                    means.append(float(col_values.mean()) if not col_values.empty else 0.0)
                    stds.append(float(col_values.std()) if len(col_values) > 1 else 0.0)
                else:
                    means.append(0.0)
                    stds.append(0.0)

        fig_div.add_trace(
            go.Scatter(
                name=legend_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                mode='lines+markers',
                marker=dict(color=model_colors.get(model_name, '#333333')),
                line=dict(color=model_colors.get(model_name, '#333333')),
                legendgroup=legend_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )

    fig_div.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    fig_div.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig_div.update_layout(
    height=350 * len(diversity_metrics),
    width=900,
    title_text="Comparison of Models: Diversity Metrics Across T Values for Dataset BH All",
    legend_title_text="Model (Temperature)"
)
fig_div.update_xaxes(title_text="T value", row=len(diversity_metrics), col=1)
fig_div.show()

metrics = [
    ('diversity_positive', 'Diversity (Positive)'),
    ('diversity_negative', 'Diversity (Negative)'),
    ('inter_diversity_negative', 'Inter-Diversity'),
]
plot_wilcoxon_diversity_results(best_models, model_names, T, metrics, alpha=0.05)

--- 🔬 Generating Wilcoxon Test Result Matrices for Diversity Metrics ---


/scratch/site/u/tarricol/conda/envs/AI_RCP_env_plots/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning:

invalid value encountered in scalar divide



In [14]:
bh_pos_random_baseline_freq_chain.head(10)

,run_idx,T,baseline,split,accuracy,macro_recall,micro_recall
3,0,1,freq_chain,pos,0.035714,0.000616,0.000614
7,0,10,freq_chain,pos,0.285714,0.011662,0.009515
11,0,50,freq_chain,pos,0.678571,0.062165,0.039902
15,0,100,freq_chain,pos,0.714286,0.096092,0.084101
19,0,500,freq_chain,pos,0.892857,0.342493,0.250767
23,0,1000,freq_chain,pos,0.928571,0.477871,0.340393
27,1,1,freq_chain,pos,0.071429,0.001064,0.001535
31,1,10,freq_chain,pos,0.214286,0.011454,0.010129
35,1,50,freq_chain,pos,0.678571,0.048645,0.034991
39,1,100,freq_chain,pos,0.714286,0.090637,0.077655


In [15]:
T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go

# Define the models and their best results (inverted order)
model_names = ['Freq Chain (pos)', 'Freq Chain (all)', 'Baseline (pos)', 'Baseline (all)', 'Seq+Emb (pos)', 'Seq+Emb (all)']
best_models = [bh_pos_random_baseline_freq_chain, bh_all_random_baseline_freq_chain, bh_pos_baseline, bh_all_baseline, bh_pos_seq_emb, bh_all_seq_emb]
model_keys = ['bh_pos_random_baseline_freq_chain', 'bh_all_random_baseline_freq_chain', 'bh_pos_baseline', 'bh_all_baseline', 'bh_pos_seq_emb', 'bh_all_seq_emb']
model_colors = {
    'Seq+Emb (pos)': '#b87053',
    'Seq+Emb (all)': '#383635',
    'Freq Chain (pos)': '#f5e2da',
    'Freq Chain (all)': '#d1cac7',
    'Baseline (pos)': '#e6ac95',
    'Baseline (all)': '#787573',
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)')
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Freq Chain (pos)', 'Freq Chain (all)']:
                try:
                    model_T = model[model['T'] == t]
                    if metric[-8:] == 'positive':
                        mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                        std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                except KeyError:
                    mean_val = 0
                    std_val = 0
            else:
                metric_col_name = f'{metric}_T{t}'
                if model_names[m_idx] == 'RxnFP':
                    # RxnFP only has T=1, so for other T set to 0
                    if t == 1:
                        mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                        std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
                    else:
                        mean_val = 0
                        std_val = 0
                else:
                    mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                    std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)

# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.08,
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=350 * n_metrics,
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for BH dataset",
    legend_title_text="Model"
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


### Metrics for embedding models

In [16]:
T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go

# Define the models and their best results (inverted order)
model_names = [ 'Baseline', 'XTB', 'SASA', 'XTB+SASA', 'CB', 'Emb']
best_models = [bh_all_baseline, bh_all_emb_xtb, bh_all_emb_sasa, bh_all_emb_sec, bh_all_emb_cb, bh_all_emb_all]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['bh_all_baseline', 'bh_all_emb_xtb', 'bh_all_emb_sasa', 'bh_all_emb_sec', 'bh_all_emb_cb', 'bh_all_emb_all']

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d',
    'SASA': '#e085fc',
    'XTB+SASA': '#bc36f0',
    'CB': '#7d0096',
    'XTB': '#f2d4ff',
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('accuracy_negative', 'Accuracy (Neg)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)'),
    ('micro_recall_negative', 'Micro Recall (Neg)'),
    ('macro_recall_negative', 'Macro Recall (Neg)')
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
                model_T = model[model['T'] == t]
                if metric[-8:] == 'positive':
                    mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                elif metric[-8:] == 'negative':
                    mean_val = np.mean(model_T[model_T['split'] == 'neg'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'neg'][metric[:-9]])
                else:
                    print(f"Metric {metric} not found")
                 
            else:
                metric_col_name = f'{metric}_T{t}'
                if model_names[m_idx] == 'RxnFP':
                    # RxnFP only has T=1, so for other T set to 0
                    if t == 1:
                        mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                        std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
                    else:
                        mean_val = 0
                        std_val = 0
                else:
                    mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                    std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)



# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.05,
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=250 * n_metrics,
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for BH all dataset",
    legend_title_text="Model",
    showlegend=True
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


---
# Training comparison SM

In [17]:
sm_all_baseline = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_baseline_sm_all_20250806-121019_detailed.csv')
sm_all_emb_cb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_all_20250806-121057_detailed.csv')
sm_all_emb_sasa = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_all_20250806-125510_detailed.csv')
sm_all_emb_sec = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_all_20250806-155328_detailed.csv')
sm_all_emb_all = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_all_20250806-154135_detailed.csv')
sm_all_emb_xtb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_all_20250806-195511_detailed.csv')
sm_all_rxnfp = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_rxnfp_sm_all_20250806-121029_detailed.csv')
sm_all_seq = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_sm_all_20250806-121023_detailed.csv')
sm_all_seq_emb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_emb_sm_all_20250806-121025_detailed.csv')

sm_pos_baseline = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_baseline_sm_positive_20250807-101136_detailed.csv') 
sm_pos_emb_all = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_positive_20250806-154638_detailed.csv')
sm_pos_rxnfp = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_rxnfp_sm_positive_20250806-154501_detailed.csv')
sm_pos_seq = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_sm_positive_20250806-124944_detailed.csv')
sm_pos_seq_emb = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_emb_sm_positive_20250806-123447_detailed.csv')


temperatures_sm = {
    'sm_all_baseline': 0.821,
    'sm_all_emb_cb': 1.666,
    'sm_all_emb_sasa': 1.784,
    'sm_all_emb_sec': 1.491,
    'sm_all_emb_all': 1.238,
    'sm_all_emb_xtb': 0.951,
    'sm_all_rxnfp': 1.728,
    'sm_all_seq': 1.674,
    'sm_all_seq_emb': 1.266, 
    'sm_pos_rxnfp': 1.308,
    'sm_pos_baseline': 1.766,
    'sm_pos_emb_all': 0.762,
    'sm_pos_seq': 1.410,
    'sm_pos_seq_emb': 0.809,
}


In [18]:
#Load random-frequency results
sm_all_random_baselines = pd.read_csv('/home/tarricol/AI_RCP/best_results/baselines_baseline_sm_all_20250807-165336.csv')
sm_positive_random_baselines = pd.read_csv('/home/tarricol/AI_RCP/best_results/baselines_baseline_sm_positive_20250807-165043.csv')

sm_all_random_baseline_random = sm_all_random_baselines[(sm_all_random_baselines['baseline'] == 'random')]
sm_all_random_baseline_structured_random = sm_all_random_baselines[(sm_all_random_baselines['baseline'] == 'structured_random')]
sm_all_random_baseline_most_frequent = sm_all_random_baselines[(sm_all_random_baselines['baseline'] == 'most_frequent')]
sm_all_random_baseline_freq_chain = sm_all_random_baselines[(sm_all_random_baselines['baseline'] == 'freq_chain')]

sm_pos_random_baseline_random = sm_positive_random_baselines[(sm_positive_random_baselines['baseline'] == 'random')]
sm_pos_random_baseline_structured_random = sm_positive_random_baselines[(sm_positive_random_baselines['baseline'] == 'structured_random')]
sm_pos_random_baseline_most_frequent = sm_positive_random_baselines[(sm_positive_random_baselines['baseline'] == 'most_frequent')]
sm_pos_random_baseline_freq_chain = sm_positive_random_baselines[(sm_positive_random_baselines['baseline'] == 'freq_chain')]

In [19]:
sm_pos_random_baseline_freq_chain

,run_idx,T,baseline,split,accuracy,macro_recall,micro_recall
3,0,1,freq_chain,pos,0.1250,0.003125,0.001846
7,0,10,freq_chain,pos,0.3125,0.004945,0.006275
11,0,50,freq_chain,pos,0.6250,0.038239,0.031377
15,0,100,freq_chain,pos,0.8125,0.069848,0.063492
19,0,500,freq_chain,pos,0.9375,0.257571,0.234035
23,0,1000,freq_chain,pos,1.0000,0.363013,0.330380
27,1,1,freq_chain,pos,0.1250,0.009370,0.001477
31,1,10,freq_chain,pos,0.1875,0.003652,0.004061
35,1,50,freq_chain,pos,0.6250,0.036574,0.043189
39,1,100,freq_chain,pos,0.8125,0.077423,0.066445


In [20]:
sm_all_baseline

,run_index,overall_performance,accuracy_positive_T1,macro_recall_positive_T1,micro_recall_positive_T1,diversity_positive_T1,accuracy_negative_T1,macro_recall_negative_T1,micro_recall_negative_T1,diversity_negative_T1,inter_diversity_negative_T1,accuracy_positive_T10,macro_recall_positive_T10,micro_recall_positive_T10,diversity_positive_T10,accuracy_negative_T10,macro_recall_negative_T10,micro_recall_negative_T10,diversity_negative_T10,inter_diversity_negative_T10,accuracy_positive_T50,macro_recall_positive_T50,micro_recall_positive_T50,diversity_positive_T50,accuracy_negative_T50,macro_recall_negative_T50,micro_recall_negative_T50,diversity_negative_T50,inter_diversity_negative_T50,accuracy_positive_T100,macro_recall_positive_T100,micro_recall_positive_T100,diversity_positive_T100,accuracy_negative_T100,macro_recall_negative_T100,micro_recall_negative_T100,diversity_negative_T100,inter_diversity_negative_T100,accuracy_positive_T500,macro_recall_positive_T500,micro_recall_positive_T500,diversity_positive_T500,accuracy_negative_T500,macro_recall_negative_T500,micro_recall_negative_T500,diversity_negative_T500,inter_diversity_negative_T500,accuracy_positive_T1000,macro_recall_positive_T1000,micro_recall_positive_T1000,diversity_positive_T1000,accuracy_negative_T1000,macro_recall_negative_T1000,micro_recall_negative_T1000,diversity_negative_T1000,inter_diversity_negative_T1000
0,0,0.478586,0.111111,0.004507,0.003971,1.0,0.10,0.000854,0.002293,1.0,1.0,0.277778,0.011040,0.016545,9.722222,0.40,0.009893,0.013184,9.75,9.75,0.555556,0.058118,0.078756,45.333333,0.75,0.063838,0.049584,45.75,45.75,0.666667,0.088534,0.121774,86.277778,0.85,0.099035,0.085125,85.90,85.90,0.777778,0.206970,0.260754,325.166667,0.90,0.221349,0.213815,325.65,325.65,0.833333,0.262478,0.324289,534.666667,0.90,0.274830,0.276584,536.60,536.60
1,1,0.488222,0.055556,0.003333,0.001985,1.0,0.10,0.000809,0.001720,1.0,1.0,0.388889,0.020643,0.017207,9.888889,0.35,0.009620,0.010318,9.95,9.95,0.666667,0.054122,0.054931,47.500000,0.90,0.068016,0.049871,45.85,45.85,0.722222,0.091979,0.091330,89.000000,0.90,0.108087,0.088277,86.35,86.35,0.777778,0.205010,0.238915,356.833333,0.90,0.269584,0.231012,339.70,339.70,0.777778,0.281762,0.348114,602.500000,0.90,0.327609,0.294067,576.65,576.65
2,2,0.431325,0.000000,0.000000,0.000000,1.0,0.15,0.004969,0.002293,1.0,1.0,0.388889,0.011416,0.011913,9.888889,0.35,0.008029,0.005732,9.95,9.95,0.666667,0.040458,0.045003,47.666667,0.70,0.028767,0.027802,47.20,47.20,0.666667,0.058383,0.068829,90.944444,0.80,0.071378,0.054743,89.85,89.85,0.777778,0.166955,0.187955,355.000000,0.90,0.158496,0.149900,349.45,349.45,0.777778,0.209787,0.258769,599.055556,0.90,0.229235,0.212382,587.90,587.90
3,3,0.508333,0.000000,0.000000,0.000000,1.0,0.10,0.001089,0.001433,1.0,1.0,0.222222,0.005902,0.008604,9.833333,0.45,0.013540,0.011465,9.75,9.75,0.555556,0.051310,0.056916,45.222222,0.80,0.056508,0.055317,44.20,44.20,0.611111,0.099845,0.107214,82.888889,0.90,0.115569,0.085125,81.10,81.10,0.833333,0.242058,0.252813,292.333333,0.95,0.268704,0.227286,281.10,281.10,0.833333,0.314227,0.321641,471.666667,1.00,0.307940,0.272858,443.50,443.50
4,4,0.493163,0.111111,0.004156,0.003309,1.0,0.05,0.000288,0.000860,1.0,1.0,0.444444,0.020265,0.024487,9.444444,0.45,0.010459,0.010891,9.75,9.75,0.666667,0.080280,0.076770,44.277778,0.80,0.081331,0.056463,44.35,44.35,0.722222,0.113155,0.116479,82.555556,0.80,0.108221,0.081112,81.40,81.40,0.777778,0.244348,0.247518,294.888889,0.90,0.262862,0.198051,289.50,289.50,0.833333,0.304981,0.313700,481.444444,0.95,0.306462,0.250502,468.20,468.20
5,5,0.518083,0.055556,0.001174,0.001985,1.0,0.10,0.001239,0.001146,1.0,1.0,0.500000,0.018837,0.015884,9.777778,0.30,0.007795,0.005446,9.85,9.85,0.611111,0.052980,0.061549,47.166667,0.75,0.066843,0.039553,45.75,45.75,0.666667,0.085960,0.108537,89.555556,0.85,0.102702,0.077386,87.30,87.30,0.777778,0.229590,0.266049,343.222222,0.95,0.283567,0.229292,331.10,331.10,0.833333,0.322749,0.358703,568.444444,0.95,0.340477

In [21]:
T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go

# Define the models and their best results (inverted order)
model_names = ['Random','RxnFP', 'Structured Random', 'Most Frequent','Frequency Chain', 'Baseline', 'Emb', 'Seq', 'Seq+Emb']
best_models = [sm_all_random_baseline_random, sm_all_rxnfp, sm_all_random_baseline_structured_random, sm_all_random_baseline_most_frequent, sm_all_random_baseline_freq_chain, sm_all_baseline, sm_all_emb_all, sm_all_seq, sm_all_seq_emb]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['sm_all_random_baseline_random', 'sm_all_rxnfp', 'sm_all_random_baseline_structured_random', 'sm_all_random_baseline_most_frequent', 'sm_all_random_baseline_freq_chain', 'sm_all_baseline', 'sm_all_emb_all', 'sm_all_seq', 'sm_all_seq_emb']

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d'
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('accuracy_negative', 'Accuracy (Neg)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)'),
    ('micro_recall_negative', 'Micro Recall (Neg)'),
    ('macro_recall_negative', 'Macro Recall (Neg)')
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
                model_T = model[model['T'] == t]
                if metric[-8:] == 'positive':
                    mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                elif metric[-8:] == 'negative':
                    mean_val = np.mean(model_T[model_T['split'] == 'neg'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'neg'][metric[:-9]])
                else:
                    print(f"Metric {metric} not found")
                    
            else:
                metric_col_name = f'{metric}_T{t}'
                if model_names[m_idx] == 'RxnFP':
                    # RxnFP only has T=1, so for other T set to 0
                    if t == 1:
                        mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                        std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
                    else:
                        mean_val = 0
                        std_val = 0
                else:
                    mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                    std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)

# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.05,
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=250 * n_metrics,
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for SM all dataset",
    legend_title_text="Model",
    showlegend=True
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


/scratch/site/u/tarricol/conda/envs/AI_RCP_env_plots/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning:

invalid value encountered in scalar divide



In [22]:

T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go
import numpy as np

# Define the models and their best results (inverted order)
model_names = ['Random','RxnFP', 'Structured Random', 'Most Frequent','Frequency Chain', 'Baseline', 'Emb', 'Seq', 'Seq+Emb']
best_models = [sm_pos_random_baseline_random, sm_pos_rxnfp, sm_pos_random_baseline_structured_random, sm_pos_random_baseline_most_frequent, sm_pos_random_baseline_freq_chain, sm_pos_baseline, sm_pos_emb_all, sm_pos_seq, sm_pos_seq_emb]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['sm_pos_random_baseline_random', 'sm_pos_rxnfp', 'sm_pos_random_baseline_structured_random', 'sm_pos_random_baseline_most_frequent', 'sm_pos_random_baseline_freq_chain', 'sm_pos_baseline', 'sm_pos_emb_all', 'sm_pos_seq', 'sm_pos_seq_emb']

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d'
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)'),
]

temperatures_sm = {
    'sm_all_baseline': 0.821,
    'sm_all_emb_cb': 1.666,
    'sm_all_emb_sasa': 1.784,
    'sm_all_emb_sec': 1.491,
    'sm_all_emb_all': 1.238,
    'sm_all_emb_xtb': 0.951,
    'sm_all_rxnfp': 1.728,
    'sm_all_seq': 1.674,
    'sm_all_seq_emb': 1.266, 
    'sm_pos_rxnfp': 1.308,
    'sm_pos_baseline': 1.766,
    'sm_pos_emb_all': 0.762,
    'sm_pos_seq': 1.410,
    'sm_pos_seq_emb': 0.809,
}

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
                try:
                    model_T = model[model['T'] == t]
                    if metric[-8:] == 'positive':
                        mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                        std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                except KeyError:
                    mean_val = 0
                    std_val = 0
            
            elif model_names[m_idx] == 'RxnFP':
                # RxnFP only has T=1, so for other T set to 0
                if t == 1:
                    mean_val = np.mean(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
                    std_val = np.std(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
                else:
                    mean_val = 0
                    std_val = 0
            else:
                mean_val = np.mean(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
                std_val = np.std(model[f'{metric}_T{t}']) if f'{metric}_T{t}' in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)

# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.15, # Reduced vertical spacing
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=250 * n_metrics, #Reduced height
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for SM positive dataset",
    legend_title_text="Model",
    showlegend=True
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


In [23]:
# Line plot for diversity_pos and diversity_neg with uncertainties

import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Define new models because there was an error in the previous code
sm_all_rxnfp_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_rxnfp_sm_all_20250812-195131_detailed.csv')
sm_all_baseline_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_baseline_sm_all_20250812-195123_detailed.csv')
sm_all_emb_all_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_emb_sm_all_20250812-195158_detailed.csv')
sm_all_seq_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_sm_all_20250812-195138_detailed.csv')
sm_all_seq_emb_new = pd.read_csv('/home/tarricol/AI_RCP/best_results/results_seq_emb_sm_all_20250812-195131_detailed.csv')

data_freq_baselines = pd.read_csv('/home/tarricol/AI_RCP/experiment_4_results/experiment4_emb_sm_all_20250813-093120.csv')

# Define the models and their best results (inverted order)
model_names = ['RxnFP', 'Baseline', 'Emb', 'Seq', 'Seq+Emb', 'Frequency Chain']
best_models = [sm_all_rxnfp_new, sm_all_baseline_new, sm_all_emb_all_new, sm_all_seq_new, sm_all_seq_emb_new, data_freq_baselines]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['sm_all_rxnfp_new', 'sm_all_baseline_new', 'sm_all_emb_all_new', 'sm_all_seq_new', 'sm_all_seq_emb_new', 'data_freq_baselines']

T = [1, 10, 50, 100, 500, 1000]

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d'
}


n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

temperatures_sm = {
    'sm_all_baseline_new': 0.821,
    'sm_all_emb_cb': 1.666,
    'sm_all_emb_sasa': 1.784,
    'sm_all_emb_sec': 1.491,
    'sm_all_emb_all_new': 1.238,
    'sm_all_emb_xtb': 0.951,
    'sm_all_rxnfp_new': 1.728,
    'sm_all_seq_new': 1.674,
    'sm_all_seq_emb_new': 1.266, 
    'sm_pos_rxnfp': 1.308,
    'sm_pos_baseline': 1.766,
    'sm_pos_emb_all': 0.762,
    'sm_pos_seq': 1.410,
    'sm_pos_seq_emb': 0.809,
}


# Metric mapping
# - 'diversity_positive' comes from rows with split == 'pos' and column 'avg_diversity_pos'
# - 'diversity_negative' comes from rows with split == 'neg' and column 'avg_diversity_neg'
# - 'inter_diversity_negative' uses column 'avg_inter_diversity' available on 'pos' rows
metric_to_freqchain_source = {
    'diversity_positive': ('avg_diversity_pos', 'pos'),
    'diversity_negative': ('avg_diversity_neg', 'neg'),
    'inter_diversity_negative': ('avg_inter_diversity', 'pos'),
}

# Display names
diversity_metrics = ['diversity_positive', 'diversity_negative', 'inter_diversity_negative']
diversity_metric_names = ['Diversity (Positive)', 'Diversity (Negative)', 'Inter-Diversity']

# Helper to compute mean/std for Frequency Chain for a given metric and T
def compute_freq_chain_stats(df: pd.DataFrame, metric_key: str, t_value: int) -> tuple[float, float]:
    col_name, required_split = metric_to_freqchain_source[metric_key]
    subset = df[(df['baseline'] == 'freq_chain') & (df['split'] == required_split) & (df['T'] == t_value)]
    values = subset[col_name].dropna()
    if values.empty:
        return 0.0, 0.0
    if len(values) == 1:
        return float(values.iloc[0]), 0.0
    return float(values.mean()), float(values.std())

# Build figure
fig_div = make_subplots(rows=3, cols=1, vertical_spacing=0.12, subplot_titles=diversity_metric_names)

for idx, (metric, metric_name) in enumerate(zip(diversity_metrics, diversity_metric_names)):
    for m_idx, model_name in enumerate(model_names):
        model_df = best_models[m_idx]
        model_key = model_keys[m_idx]
        means: list[float] = []
        stds: list[float] = []

        temperature = round(temperatures_sm.get(model_key, None), 2) if model_key in temperatures_sm else None
        legend_name = f"{model_name} ({temperature})" if temperature is not None else model_name

        if model_name == 'Frequency Chain':
            for t in T:
                mean_val, std_val = compute_freq_chain_stats(model_df, metric, t)
                means.append(mean_val)
                stds.append(std_val)
        elif model_name.lower() == 'rxnfp':
            means = [1.0 for _ in T]
            stds = [0.0 for _ in T]
        else:
            for t in T:
                metric_col_name = f'{metric}_T{t}'
                if metric_col_name in model_df.columns:
                    col_values = model_df[metric_col_name].dropna()
                    means.append(float(col_values.mean()) if not col_values.empty else 0.0)
                    stds.append(float(col_values.std()) if len(col_values) > 1 else 0.0)
                else:
                    means.append(0.0)
                    stds.append(0.0)

        fig_div.add_trace(
            go.Scatter(
                name=legend_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                mode='lines+markers',
                marker=dict(color=model_colors.get(model_name, '#333333')),
                line=dict(color=model_colors.get(model_name, '#333333')),
                legendgroup=legend_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )

    fig_div.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    fig_div.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig_div.update_layout(
    height=350 * len(diversity_metrics),
    width=900,
    title_text="Comparison of Models: Diversity Metrics Across T Values for Dataset SM All",
    legend_title_text="Model (Temperature)"
)
fig_div.update_xaxes(title_text="T value", row=len(diversity_metrics), col=1)
fig_div.show()

metrics = [
    ('diversity_positive', 'Diversity (Positive)'),
    ('diversity_negative', 'Diversity (Negative)'),
    ('inter_diversity_negative', 'Inter-Diversity'),
]
plot_wilcoxon_diversity_results(best_models, model_names, T, metrics, alpha=0.05)

--- 🔬 Generating Wilcoxon Test Result Matrices for Diversity Metrics ---


/scratch/site/u/tarricol/conda/envs/AI_RCP_env_plots/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning:

invalid value encountered in scalar divide



In [24]:
bh_pos_random_baseline_freq_chain.head(10)

,run_idx,T,baseline,split,accuracy,macro_recall,micro_recall
3,0,1,freq_chain,pos,0.035714,0.000616,0.000614
7,0,10,freq_chain,pos,0.285714,0.011662,0.009515
11,0,50,freq_chain,pos,0.678571,0.062165,0.039902
15,0,100,freq_chain,pos,0.714286,0.096092,0.084101
19,0,500,freq_chain,pos,0.892857,0.342493,0.250767
23,0,1000,freq_chain,pos,0.928571,0.477871,0.340393
27,1,1,freq_chain,pos,0.071429,0.001064,0.001535
31,1,10,freq_chain,pos,0.214286,0.011454,0.010129
35,1,50,freq_chain,pos,0.678571,0.048645,0.034991
39,1,100,freq_chain,pos,0.714286,0.090637,0.077655


In [25]:
T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go

# Define the models and their best results (inverted order)
model_names = ['Freq Chain (pos)', 'Freq Chain (all)', 'Baseline (pos)', 'Baseline (all)', 'Seq+Emb (pos)', 'Seq+Emb (all)']
best_models = [sm_pos_random_baseline_freq_chain, sm_all_random_baseline_freq_chain, sm_pos_baseline, sm_all_baseline, sm_pos_seq_emb, sm_all_seq_emb]
model_keys = ['sm_pos_random_baseline_freq_chain', 'sm_all_random_baseline_freq_chain', 'sm_pos_baseline', 'sm_all_baseline', 'sm_pos_seq_emb', 'sm_all_seq_emb']
model_colors = {
    'Seq+Emb (pos)': '#b87053',
    'Seq+Emb (all)': '#383635',
    'Freq Chain (pos)': '#f5e2da',
    'Freq Chain (all)': '#d1cac7',
    'Baseline (pos)': '#e6ac95',
    'Baseline (all)': '#787573',
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)')
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Freq Chain (pos)', 'Freq Chain (all)']:
                try:
                    model_T = model[model['T'] == t]
                    if metric[-8:] == 'positive':
                        mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                        std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                except KeyError:
                    mean_val = 0
                    std_val = 0
            else:
                metric_col_name = f'{metric}_T{t}'
                if model_names[m_idx] == 'RxnFP':
                    # RxnFP only has T=1, so for other T set to 0
                    if t == 1:
                        mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                        std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
                    else:
                        mean_val = 0
                        std_val = 0
                else:
                    mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                    std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)

# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.08,
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=350 * n_metrics,
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for SM dataset",
    legend_title_text="Model"
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---


### Metrics for embedding models

In [26]:
T = [1, 10, 50, 100, 500, 1000]

import plotly.graph_objects as go

# Define the models and their best results (inverted order)
model_names = [ 'Baseline', 'XTB', 'SASA', 'XTB+SASA', 'CB', 'Emb']
best_models = [sm_all_baseline, sm_all_emb_xtb, sm_all_emb_sasa, sm_all_emb_sec, sm_all_emb_cb, sm_all_emb_all]
# --- FIX: Create a list of the dictionary keys in the same order ---
model_keys = ['sm_all_baseline', 'sm_all_emb_xtb', 'sm_all_emb_sasa', 'sm_all_emb_sec', 'sm_all_emb_cb', 'sm_all_emb_all']

model_colors = {
    'Seq+Emb': '#022366',
    'Seq': '#0b41cd',
    'Emb': '#1482fa',
    'Baseline': '#bde3ff',
    'RxnFP': '#ff8782', 
    'Random': '#ffbd69',
    'Structured Random': '#ff7d29',
    'Most Frequent': '#ed4a0d',
    'Frequency Chain': '#b22b0d',
    'SASA': '#e085fc',
    'XTB+SASA': '#bc36f0',
    'CB': '#7d0096',
    'XTB': '#f2d4ff',
}

# Define the metrics and their pretty names
metrics = [
    ('accuracy_positive', 'Accuracy (Pos)'),
    ('accuracy_negative', 'Accuracy (Neg)'),
    ('micro_recall_positive', 'Micro Recall (Pos)'),
    ('macro_recall_positive', 'Macro Recall (Pos)'),
    ('micro_recall_negative', 'Micro Recall (Neg)'),
    ('macro_recall_negative', 'Macro Recall (Neg)')
]

# Prepare the data for plotting
mean_data = {metric: [] for metric, _ in metrics}
std_data = {metric: [] for metric, _ in metrics}

for metric, _ in metrics:
    for m_idx, model in enumerate(best_models):
        means = []
        stds = []
        for t in T:
            if model_names[m_idx] in ['Random', 'Structured Random', 'Most Frequent', 'Frequency Chain']:
                model_T = model[model['T'] == t]
                if metric[-8:] == 'positive':
                    mean_val = np.mean(model_T[model_T['split'] == 'pos'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'pos'][metric[:-9]])
                elif metric[-8:] == 'negative':
                    mean_val = np.mean(model_T[model_T['split'] == 'neg'][metric[:-9]])
                    std_val = np.std(model_T[model_T['split'] == 'neg'][metric[:-9]])
                else:
                    print(f"Metric {metric} not found")
                 
            else:
                metric_col_name = f'{metric}_T{t}'
                if model_names[m_idx] == 'RxnFP':
                    # RxnFP only has T=1, so for other T set to 0
                    if t == 1:
                        mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                        std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
                    else:
                        mean_val = 0
                        std_val = 0
                else:
                    mean_val = np.mean(model[metric_col_name]) if metric_col_name in model else 0
                    std_val = np.std(model[metric_col_name]) if metric_col_name in model else 0
            means.append(mean_val)
            stds.append(std_val)
        mean_data[metric].append(means)
        std_data[metric].append(stds)



# Plotting with plotly
from plotly.subplots import make_subplots

n_metrics = len(metrics)
n_models = len(model_names)
n_T = len(T)
x_tick_labels = [f"T={t}" for t in T]

fig = make_subplots(
    rows=n_metrics, cols=1,
    vertical_spacing=0.05,
    subplot_titles=[pretty for _, pretty in metrics]
)

for idx, (metric, metric_name) in enumerate(metrics):
    for m_idx, model_name in enumerate(model_names):
        means = mean_data[metric][m_idx]
        stds = std_data[metric][m_idx]
        fig.add_trace(
            go.Bar(
                name=model_name,
                x=x_tick_labels,
                y=means,
                error_y=dict(type='data', array=stds, visible=True),
                marker_color=model_colors[model_name],
                offsetgroup=m_idx,
                legendgroup=model_name,
                showlegend=(idx == 0)
            ),
            row=idx+1, col=1
        )
    fig.update_yaxes(title_text=metric_name, row=idx+1, col=1)
    # Set x-tick labels for every subplot (no x-axis title)
    fig.update_xaxes(
        title_text="T value",
        tickmode='array',
        tickvals=x_tick_labels,
        ticktext=x_tick_labels,
        row=idx+1, col=1
    )

fig.update_layout(
    barmode='group',
    height=250 * n_metrics,
    width=900,
    title_text="Comparison of Models Across Metrics and T Values for SM all dataset",
    legend_title_text="Model",
    showlegend=True
)
# Only add x-axis title to the bottom plot
fig.update_xaxes(title_text="T value", row=n_metrics, col=1)
fig.show()

# --- Perform the Statistical Tests ---
plot_wilcoxon_results(best_models, model_names, T, metrics)

--- 🔬 Generating Wilcoxon Test Result Matrices ---
